In [25]:
import setuptools
import os
import re
import string
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import scipy.sparse

import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

# ========================== CONFIGURATION ==========================
CONFIG = {
    "data_path":r"C:\practice\Text_Classification_ML_pipeline\notebooks\data.csv",
    "test_size" : 0.2,
    "mlflow_tracking_uri": "https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow",
    "dagshub_repo_owner":'Shrutichauha7',
    "dagshub_repo_name": 'Text_Classification_ML_pipeline',
    "experiment_name": "Bow vs TfIdf"

}

# ========================== SETUP MLflow & DAGSHUB ==========================
mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
mlflow.set_experiment(CONFIG["experiment_name"])

# ========================== TEXT PREPROCESSING ==========================
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    return " ".join([word for word in text.split() if word not in stop_words])

def removing_numbers(text):
    return ''.join([char for char in text if not char.isdigit()])

def lower_case(text):
    return text.lower()

def removing_punctuations(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", ' ', text)

def removing_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def normalize_text(df):
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

# ========================== LOAD & PREPROCESS DATA ==========================
def load_data(file_path):
    try:
        df = pd.read_csv(file_path)
        df = normalize_text(df)
        df = df[df['sentiment'].isin(['positive', 'negative'])]
        df['sentiment'] = df['sentiment'].replace({'negative': 0, 'positive': 1}).infer_objects(copy=False)
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

# ========================== FEATURE ENGINEERING ==========================
VECTORIZERS = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

# ========================== TRAIN & EVALUATE MODELS ==========================
def train_and_evaluate(df):
    with mlflow.start_run(run_name="All Experiments") as parent_run:
        for algo_name, algorithm in ALGORITHMS.items():
            for vec_name, vectorizer in VECTORIZERS.items():
                with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                    try:
                        # Feature extraction
                        X = vectorizer.fit_transform(df['review'])
                        y = df['sentiment']
                        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=CONFIG["test_size"], random_state=42)

                        # Log preprocessing parameters
                        mlflow.log_params({
                            "vectorizer": vec_name,
                            "algorithm": algo_name,
                            "test_size": CONFIG["test_size"]
                        })

                        # Train model
                        model = algorithm
                        model.fit(X_train, y_train)

                        # Log model parameters
                        log_model_params(algo_name, model)

                        # Evaluate model
                        y_pred = model.predict(X_test)
                        metrics = {
                            "accuracy": accuracy_score(y_test, y_pred),
                            "precision": precision_score(y_test, y_pred),
                            "recall": recall_score(y_test, y_pred),
                            "f1_score": f1_score(y_test, y_pred)
                        }
                        mlflow.log_metrics(metrics)

                        # Log model
                        # mlflow.sklearn.log_model(model, "model")
                        input_example = X_test[:5] if not scipy.sparse.issparse(X_test) else X_test[:5].toarray()
                        mlflow.sklearn.log_model(model, "model", input_example=input_example)

                        # Print results for verification
                        print(f"\nAlgorithm: {algo_name}, Vectorizer: {vec_name}")
                        print(f"Metrics: {metrics}")

                    except Exception as e:
                        print(f"Error in training {algo_name} with {vec_name}: {e}")
                        mlflow.log_param("error", str(e))

def log_model_params(algo_name, model):
    """Logs hyperparameters of the trained model to MLflow."""
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'XGBoost':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
    elif algo_name == 'RandomForest':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    elif algo_name == 'GradientBoosting':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
        params_to_log["max_depth"] = model.max_depth

    mlflow.log_params(params_to_log)

# ========================== EXECUTION ==========================
if __name__ == "__main__":
    df = load_data(CONFIG["data_path"])
    train_and_evaluate(df)

Initialized MLflow to track repo "Shrutichauha7/Text_Classification_ML_pipeline"

Repository Shrutichauha7/Text_Classification_ML_pipeline initialized!

2026/06/17 22:39:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 22:39:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: LogisticRegression, Vectorizer: BoW
Metrics: {'accuracy': 0.75, 'precision': 0.7777777777777778, 'recall': 0.7, 'f1_score': 0.7368421052631579}
🏃 View run LogisticRegression with BoW at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1/runs/bded309e39804d56801e1f0f42980211
🧪 View experiment at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1


2026/06/17 22:40:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 22:40:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...


Algorithm: RandomForest, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.72, 'precision': 0.84375, 'recall': 0.54, 'f1_score': 0.6585365853658537}
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1/runs/ac48a15cf45d4c3388f92c3196266bf3
🧪 View experiment at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1


2026/06/17 22:44:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 22:44:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: GradientBoosting, Vectorizer: BoW
Metrics: {'accuracy': 0.75, 'precision': 0.7906976744186046, 'recall': 0.68, 'f1_score': 0.7311827956989247}
🏃 View run GradientBoosting with BoW at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1/runs/9b6579ebae204ea3b1ceb056aaa14a67
🧪 View experiment at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1


2026/06/17 22:44:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 22:44:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: GradientBoosting, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.69, 'precision': 0.6792452830188679, 'recall': 0.72, 'f1_score': 0.6990291262135923}
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1/runs/336c9e00906e47c280bb5dee4b97d24c
🧪 View experiment at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1
🏃 View run All Experiments at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1/runs/71936753ac88479ea1e79821d54747ee
🧪 View experiment at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/1
